# 02 — Types of Training

**Learning objectives**
- Understand activation functions and why they matter
- Build a linear regression model from scratch and understand gradient descent
- Know what loss functions measure and how to evaluate a model
- Distinguish supervised, unsupervised, and reinforcement learning — with a working example of each

**Prerequisites:** Notebook 01 — What is AI?

---

## 1. The Perceptron's Intrinsic Function: Activation

In notebook 01, our perceptron fired if the weighted sum exceeded a threshold — a hard on/off. Real neural networks replace that hard switch with an **activation function**: a mathematical operation applied to the weighted sum before passing it to the next layer.

Why? Two reasons:
1. **Non-linearity** — without it, stacking layers is mathematically equivalent to a single layer. The network can't learn complex patterns.
2. **Differentiability** — we need to compute gradients during training (more on this shortly). A hard step function has a gradient of zero almost everywhere, which breaks learning.

### The three you'll see everywhere

**Sigmoid** — squashes any number into (0, 1). Good for probabilities, but causes "vanishing gradients" in deep networks.
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**ReLU** (Rectified Linear Unit) — outputs the input if positive, zero otherwise. Simple, fast, and the default choice for hidden layers today.
$$\text{ReLU}(x) = \max(0, x)$$

**Softmax** — generalises sigmoid to multiple classes. Converts a vector of scores into a probability distribution that sums to 1. Used in the output layer for classification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x): return 1 / (1 + np.exp(-x))
def relu(x):    return np.maximum(0, x)
def step(x):    return (x > 0).astype(float)

x = np.linspace(-6, 6, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, fn, name, color in zip(
    axes,
    [step, sigmoid, relu],
    ["Step (original perceptron)", "Sigmoid", "ReLU"],
    ["gray", "steelblue", "tomato"]
):
    ax.plot(x, fn(x), color=color, linewidth=2.5)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("weighted sum (z)")
    ax.set_ylabel("activation output")
    ax.grid(True, alpha=0.3)

plt.suptitle("Activation Functions", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print("Key takeaway: Step has no gradient. Sigmoid saturates at extremes.")
print("ReLU is simple and works well — it's the default for hidden layers.")

## 2. Learning from Data: Linear Regression

Before tackling neural networks, let's understand the training mechanism using the simplest possible model: **linear regression**.

The goal: given some input `x`, predict an output `y` using a straight line.

$$\hat{y} = w \cdot x + b$$

- `w` (weight) — the slope of the line
- `b` (bias) — where the line crosses the y-axis

Initially `w` and `b` are random guesses. Training is the process of adjusting them until the line fits the data.

### The Loss Function

We need a way to measure how wrong our predictions are. For regression, we use **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

Squaring the errors penalises large mistakes heavily and keeps everything positive.

### Gradient Descent

To minimise the loss, we compute how much each parameter (`w` and `b`) contributed to the error — that's the **gradient** — and nudge them in the opposite direction.

$$w \leftarrow w - \alpha \cdot \frac{\partial \text{Loss}}{\partial w}$$

`α` (alpha) is the **learning rate**: how big a step to take. Too large and you overshoot; too small and training is painfully slow.

In [ ]:
np.random.seed(0)

# Ground truth: y = 2x + 1, with some noise
x = np.linspace(0, 10, 50)
y = 2 * x + 1 + np.random.randn(50) * 2

# Start with a random guess
w, b = 0.0, 0.0
lr = 0.005
loss_history = []

for epoch in range(300):
    y_pred = w * x + b
    loss = np.mean((y_pred - y) ** 2)
    loss_history.append(loss)

    # Gradients (calculus tells us these partial derivatives)
    dw = np.mean(2 * (y_pred - y) * x)
    db = np.mean(2 * (y_pred - y))

    w -= lr * dw
    b -= lr * db

print(f"Learned:  w = {w:.3f}, b = {b:.3f}")
print(f"Truth:    w = 2.000, b = 1.000")
print(f"Final loss (MSE): {loss_history[-1]:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.scatter(x, y, alpha=0.6, label="data", color="steelblue")
ax1.plot(x, w * x + b, color="tomato", linewidth=2.5, label=f"learned line (w={w:.2f}, b={b:.2f})")
ax1.set_title("Linear Regression Fit")
ax1.set_xlabel("x"); ax1.set_ylabel("y")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(loss_history, color="darkorange", linewidth=2)
ax2.set_title("Loss over Training")
ax2.set_xlabel("epoch"); ax2.set_ylabel("MSE loss")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Evaluating a Model: Accuracy, Precision, Recall, F1

Loss tells us how training is going. But once the model is trained, we need metrics that are meaningful to humans. These are classification metrics — they apply when the model is predicting a category (spam/not spam, cat/dog, etc.).

Start with a **confusion matrix**: for each example, was the prediction right, and if wrong, which way?

|  | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | True Positive (TP) | False Negative (FN) |
| **Actually Negative** | False Positive (FP) | True Negative (TN) |

From that, four metrics:

| Metric | Formula | Question it answers |
|---|---|---|
| **Accuracy** | (TP + TN) / total | How often is the model right overall? |
| **Precision** | TP / (TP + FP) | When it says positive, how often is it right? |
| **Recall** | TP / (TP + FN) | Of all real positives, how many did it catch? |
| **F1** | 2 · (P · R) / (P + R) | Harmonic mean of precision and recall |

### When does each matter?

- **Medical diagnosis** — high recall matters most. Missing a real case (FN) is worse than a false alarm.
- **Spam filter** — high precision matters most. Sending a real email to spam (FP) is worse than missing some spam.
- **F1** — use when you want a single number that balances both.

In [ ]:
def compute_metrics(y_true, y_pred):
    TP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    TN = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    FP = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    FN = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

    accuracy  = (TP + TN) / len(y_true)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"  Confusion matrix:  TP={TP}  TN={TN}  FP={FP}  FN={FN}")
    print(f"  Accuracy:   {accuracy:.2f}")
    print(f"  Precision:  {precision:.2f}  (of predicted positives, how many were real?)")
    print(f"  Recall:     {recall:.2f}  (of real positives, how many did we catch?)")
    print(f"  F1 score:   {f1:.2f}")

# Scenario A: balanced, decent model
y_true = [1,1,1,1,1, 0,0,0,0,0]
y_pred = [1,1,1,0,0, 0,0,0,1,1]
print("Scenario A — balanced classes:")
compute_metrics(y_true, y_pred)

# Scenario B: high recall, low precision (casts a wide net)
y_pred_b = [1,1,1,1,1, 1,1,0,0,0]
print("\nScenario B — high recall model (aggressive):")
compute_metrics(y_true, y_pred_b)

## 4. Types of Training

Now that we understand *how* a model learns, let's look at *what kind of signal* it learns from. There are three fundamentally different setups.

---

### Supervised Learning
The model learns from **labelled examples** — each input has a known correct answer.

- Training data: `(email text → spam / not spam)`, `(image → cat / dog)`, `(house features → price)`
- The loss function compares the model's prediction to the label and adjusts weights
- Most of AI in production today is supervised learning

---

### Unsupervised Learning
The model receives **no labels** — it finds structure in the data on its own.

- Training data: raw inputs only, no answers
- The model learns to cluster, compress, or generate data
- Examples: customer segmentation, anomaly detection, generative models

---

### Reinforcement Learning
The model (called an **agent**) learns by **interacting with an environment** and receiving rewards or penalties.

- No labelled dataset — the agent generates its own experience by taking actions
- Good actions (reaching a goal) yield positive reward; bad actions (crashing) yield negative reward
- The agent learns a **policy**: given a situation, what action maximises future reward?
- Examples: game-playing AIs, robotics, self-driving cars

| | Supervised | Unsupervised | Reinforcement |
|---|---|---|---|
| **Signal** | Human labels | None | Reward from environment |
| **Data** | Labelled dataset | Unlabelled dataset | Agent's own experience |
| **Learns** | Input → output mapping | Hidden structure | Action → long-term reward |

## 5. Supervised Learning — Example: Classifying Tumours

A classic supervised task: given two measurements from a medical scan, predict whether a tumour is malignant (1) or benign (0).

We'll train a logistic regression model — a single perceptron with a sigmoid activation — from scratch.

In [ ]:
np.random.seed(42)

# Synthetic dataset: 2 features (e.g. size, density)
# Class 0 = benign, Class 1 = malignant
n = 100
X0 = np.random.randn(n, 2) + np.array([1, 1])   # benign cluster
X1 = np.random.randn(n, 2) + np.array([3, 3])   # malignant cluster
X = np.vstack([X0, X1])
y = np.array([0] * n + [1] * n, dtype=float)

# Normalise features
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Logistic regression: perceptron + sigmoid + binary cross-entropy loss
w = np.zeros(2)
b = 0.0
lr = 0.1
loss_history = []

for epoch in range(200):
    z = X @ w + b
    y_pred = sigmoid(z)
    # Binary cross-entropy loss
    loss = -np.mean(y * np.log(y_pred + 1e-9) + (1 - y) * np.log(1 - y_pred + 1e-9))
    loss_history.append(loss)

    error = y_pred - y
    w -= lr * (X.T @ error) / len(y)
    b -= lr * error.mean()

# Evaluate
preds = (sigmoid(X @ w + b) >= 0.5).astype(int)
print("Final model evaluation on training data:")
compute_metrics(y.astype(int).tolist(), preds.tolist())

# Plot decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

x1_range = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
x2_range = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
Z = sigmoid(np.c_[xx1.ravel(), xx2.ravel()] @ w + b).reshape(xx1.shape)

ax1.contourf(xx1, xx2, Z, levels=50, cmap="RdBu_r", alpha=0.4)
ax1.scatter(X[:n, 0], X[:n, 1], label="Benign", color="steelblue", edgecolors="white", s=40)
ax1.scatter(X[n:, 0], X[n:, 1], label="Malignant", color="tomato", edgecolors="white", s=40)
ax1.set_title("Decision Boundary (Logistic Regression)")
ax1.set_xlabel("Feature 1"); ax1.set_ylabel("Feature 2")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(loss_history, color="darkorange", linewidth=2)
ax2.set_title("Training Loss (Binary Cross-Entropy)")
ax2.set_xlabel("epoch"); ax2.set_ylabel("loss")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Unsupervised Learning — Example: Customer Clustering

No labels this time. We have customer data (purchase frequency, average spend) and want to find natural groups — without being told how many or what they represent.

**K-Means** is the simplest clustering algorithm:
1. Pick `k` random points as cluster centres
2. Assign each data point to its nearest centre
3. Move each centre to the mean of its assigned points
4. Repeat until stable

The algorithm never sees a label. The groupings emerge from the geometry of the data.

In [ ]:
np.random.seed(7)

# Synthetic customer data: purchase frequency vs average spend
clusters_true = [
    np.random.randn(60, 2) + [1, 1],   # low freq, low spend
    np.random.randn(60, 2) + [5, 5],   # high freq, high spend
    np.random.randn(60, 2) + [1, 7],   # low freq, high spend (big occasional buyers)
]
data = np.vstack(clusters_true)

def kmeans(data, k=3, max_iter=100):
    # Initialise centres randomly from data points
    centres = data[np.random.choice(len(data), k, replace=False)]
    for _ in range(max_iter):
        # Assign each point to nearest centre
        dists = np.linalg.norm(data[:, None] - centres[None, :], axis=2)
        labels = dists.argmin(axis=1)
        # Recompute centres
        new_centres = np.array([data[labels == i].mean(axis=0) for i in range(k)])
        if np.allclose(centres, new_centres):
            break
        centres = new_centres
    return labels, centres

labels, centres = kmeans(data, k=3)

colors = ["steelblue", "tomato", "seagreen"]
segment_names = ["Occasional low-spenders", "Frequent high-spenders", "Big occasional buyers"]

plt.figure(figsize=(8, 6))
for i in range(3):
    pts = data[labels == i]
    plt.scatter(pts[:, 0], pts[:, 1], color=colors[i], alpha=0.6,
                label=f"Cluster {i}: {segment_names[i]}", s=40)
    plt.scatter(*centres[i], color=colors[i], marker="X", s=200, edgecolors="black", zorder=5)

plt.title("K-Means Clustering — Customer Segments\n(no labels used during training)", fontsize=13)
plt.xlabel("Purchase Frequency")
plt.ylabel("Average Spend")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The model found these groups with zero human labels.")
print("Naming the segments ('Occasional buyers' etc.) is a human job done after the fact.")

## 7. Reinforcement Learning — A Car Learning to Drive

No dataset. No labels. Just an agent, an environment, and a reward signal.

### The setup

A car drives on a 1D track (a row of cells). It needs to reach a **goal cell** without going out of bounds.

**State:** `(position, speed)` — where the car is and how fast it's going

**Actions:** 9 combinations of:
- Throttle: `accelerate` / `coast` / `brake`
- Steering: `left` / `straight` / `right`

For this 1D track, steering maps to lateral position change. Think of it as a 2D grid with rows (lateral) and columns (forward progress).

**Reward:**
- `+100` for reaching the goal
- `-10` for going out of bounds (crash)
- `-1` per step (encourages efficiency)

### Q-Learning

The agent maintains a **Q-table**: for every (state, action) pair, an estimate of how much future reward that action is worth from that state.

After each step:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \right]$$

- `r` = reward received
- `γ` (gamma) = discount factor — how much the agent values future rewards vs immediate ones
- The agent explores randomly at first (ε-greedy), then increasingly trusts its learned Q-values

In [ ]:
import random
from collections import defaultdict

# ── Environment ──────────────────────────────────────────────────────────────
GRID_ROWS  = 5   # lateral lanes (0 = far left, 4 = far right)
GRID_COLS  = 10  # track length
START      = (2, 0)   # (row, col) — middle lane, start line
GOAL_COL   = 9        # reach any cell in the last column

# Actions: (d_row, d_speed)  — steering changes lane, throttle changes speed
# Speed: 0=stopped, 1=slow, 2=fast
ACTIONS = {
    0: ("left+brake",     -1, -1),
    1: ("left+coast",     -1,  0),
    2: ("left+accel",     -1, +1),
    3: ("straight+brake",  0, -1),
    4: ("straight+coast",  0,  0),
    5: ("straight+accel",  0, +1),
    6: ("right+brake",    +1, -1),
    7: ("right+coast",    +1,  0),
    8: ("right+accel",    +1, +1),
}

def step(state, action_id):
    """Simulator: given current state and action, return (next_state, reward, done)."""
    row, col, speed = state
    _, d_row, d_speed = ACTIONS[action_id]

    new_speed = max(0, min(2, speed + d_speed))
    new_col   = col + max(1, new_speed)   # always move forward at least 1 cell
    new_row   = row + d_row

    # Out of bounds
    if new_row < 0 or new_row >= GRID_ROWS or new_col >= GRID_COLS + 3:
        return state, -10, True   # crash

    new_col = min(new_col, GRID_COLS - 1)

    # Reached goal
    if new_col >= GOAL_COL:
        return (new_row, new_col, new_speed), +100, True

    return (new_row, new_col, new_speed), -1, False

# ── Q-Learning Agent ─────────────────────────────────────────────────────────
Q = defaultdict(lambda: np.zeros(len(ACTIONS)))

alpha   = 0.1    # learning rate
gamma   = 0.95   # discount factor
epsilon = 1.0    # exploration rate (starts high, decays)

episodes        = 3000
rewards_per_ep  = []
success_count   = 0

for ep in range(episodes):
    state   = START + (1,)   # (row=2, col=0, speed=1)
    total_r = 0

    for _ in range(50):   # max steps per episode
        # ε-greedy: explore randomly or exploit best known action
        if random.random() < epsilon:
            action = random.randint(0, len(ACTIONS) - 1)
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, done = step(state, action)
        total_r += reward

        # Q-learning update
        best_next = np.max(Q[next_state])
        Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])

        state = next_state
        if done:
            if reward == 100:
                success_count += 1
            break

    rewards_per_ep.append(total_r)
    epsilon = max(0.05, epsilon * 0.998)   # decay exploration

# ── Results ───────────────────────────────────────────────────────────────────
window = 100
smoothed = np.convolve(rewards_per_ep, np.ones(window)/window, mode="valid")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(smoothed, color="steelblue", linewidth=2)
ax1.set_title(f"Average Reward per Episode (window={window})")
ax1.set_xlabel("episode"); ax1.set_ylabel("avg reward")
ax1.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax1.grid(True, alpha=0.3)

# Show the learned policy on the grid (at speed=1)
policy_grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=int)
for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        policy_grid[r, c] = int(np.argmax(Q[(r, c, 1)]))

steering_symbol = {-1: "←", 0: "↑", 1: "→"}
arrow_grid = [[steering_symbol[ACTIONS[policy_grid[r, c]][1]] for c in range(GRID_COLS)]
              for r in range(GRID_ROWS)]

ax2.axis("off")
ax2.set_title("Learned Steering Policy (at speed=1)\nRow=lane, Col=track position, Arrow=steer direction")
table = ax2.table(cellText=arrow_grid,
                  rowLabels=[f"Lane {r}" for r in range(GRID_ROWS)],
                  colLabels=[f"C{c}" for c in range(GRID_COLS)],
                  cellLoc="center", loc="center")
table.scale(1, 1.5)

plt.tight_layout()
plt.show()

print(f"Successful runs (reached goal): {success_count} / {episodes}")
print(f"Final exploration rate (ε): {epsilon:.3f}")

### 7b. Visualizing the Agent: Circular Track

The previous demo used a simple 1D grid. Let's make it real: a 2D oval track on a discrete grid, where the car has an actual **position**, **heading** (one of 8 compass directions), and **speed**.

The agent now needs to learn not just to move forward, but to **steer around corners** while staying on the track.

**Track:** an oval ring defined by two ellipses — cells between the inner and outer ellipse are driveable.
**Progress:** measured by how much of the oval the car sweeps clockwise around the center.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from collections import defaultdict
import random

# ── Track Definition ──────────────────────────────────────────────────────────
ROWS, COLS = 15, 25
CR, CC = 7, 12  # center of the oval

def on_track(r, c):
    outer = (r - CR)**2 / 6.0**2 + (c - CC)**2 / 11.0**2
    inner = (r - CR)**2 / 3.5**2 + (c - CC)**2 /  7.5**2
    return inner >= 1.0 and outer <= 1.0

track_set = frozenset((r, c) for r in range(ROWS) for c in range(COLS) if on_track(r, c))

# Start: middle of the track width on the right side (row = CR, median col among right-side cells)
right_cols = sorted(c for (r, c) in track_set if r == CR and c > CC)
START_R    = CR
START_C    = right_cols[len(right_cols) // 2]
START_STATE = (START_R, START_C, 4, 1)  # heading=4 (South), speed=1

# ── Grid image (0=off, 1=track, 2=start) ─────────────────────────────────────
cmap = mcolors.ListedColormap(['#f0f0f0', '#999999', '#44bb44', '#dd3333'])

base_grid = np.zeros((ROWS, COLS))
for (r, c) in track_set:
    base_grid[r, c] = 1.0
base_grid[START_R, START_C] = 2.0

def draw_track(ax, grid, title=""):
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=3, interpolation='nearest', aspect='equal')
    for x in np.arange(-0.5, COLS, 1):
        ax.axvline(x, color='lightgray', linewidth=0.4)
    for y in np.arange(-0.5, ROWS, 1):
        ax.axhline(y, color='lightgray', linewidth=0.4)
    ax.set_xlim(-0.5, COLS - 0.5)
    ax.set_ylim(ROWS - 0.5, -0.5)
    if title:
        ax.set_title(title, fontsize=13)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

fig, ax = plt.subplots(figsize=(13, 8))
draw_track(ax, base_grid, "Circular Track — Discrete Grid")
legend_elements = [
    Patch(facecolor='#999999', label='Track'),
    Patch(facecolor='#44bb44', label='Start / Finish line'),
    Patch(facecolor='#dd3333', label='Car'),
    Patch(facecolor='#f0f0f0', edgecolor='lightgray', label='Off-track'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

print(f"Track cells: {len(track_set)}")
print(f"Start: row={START_R}, col={START_C} | Middle of track width | Heading: South")

In [ ]:
# ── Car Model ────────────────────────────────────────────────────────────────
# 8 headings: 0=N, 1=NE, 2=E, 3=SE, 4=S, 5=SW, 6=W, 7=NW
DIR      = [(-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1), (-1,-1)]
DIR_NAME = ['N','NE','E','SE','S','SW','W','NW']

# 9 actions: (steer, throttle) — steer: -1=left / 0=straight / +1=right
#                                throttle: -1=brake / 0=coast / +1=accel
ACTIONS   = [(s, t) for s in [-1, 0, 1] for t in [-1, 0, 1]]
N_ACTIONS = len(ACTIONS)

# Arrow offsets for imshow coords (x=col right+, y=row down+)
DIR_ARROW = {
    0: ( 0.0, -0.45), 1: ( 0.45,-0.45), 2: ( 0.45, 0.0), 3: ( 0.45, 0.45),
    4: ( 0.0,  0.45), 5: (-0.45, 0.45), 6: (-0.45, 0.0), 7: (-0.45,-0.45),
}

def step_env(state, action_id):
    r, c, heading, speed = state
    steer, throttle = ACTIONS[action_id]

    new_heading = (heading + steer) % 8
    new_speed   = max(1, min(2, speed + throttle))

    dr, dc = DIR[new_heading]
    nr, nc = r + dr * new_speed, c + dc * new_speed

    if not (0 <= nr < ROWS and 0 <= nc < COLS):
        return (r, c, new_heading, new_speed), -10, True

    new_state = (nr, nc, new_heading, new_speed)

    if (nr, nc) not in track_set:
        return new_state, -10, True

    old_angle = np.arctan2(r  - CR, c  - CC)
    new_angle = np.arctan2(nr - CR, nc - CC)
    delta = new_angle - old_angle
    if delta >  np.pi: delta -= 2 * np.pi
    if delta < -np.pi: delta += 2 * np.pi

    reward = delta * 4 - 0.2
    return new_state, reward, False

print("Car model ready. Actions:", N_ACTIONS, "| Headings:", len(DIR))

### Before Training: Pure Chaos

Before any learning, the agent picks actions at random (ε = 1.0). Watch it crash almost immediately, reset, and wander aimlessly. The red flash marks each crash and reset to the start line.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

random.seed(42)

def run_random_agent(total_frames=80):
    """Run a fully random agent, resetting on crash. Returns trajectory + crash frame indices."""
    state, traj, crashes = START_STATE, [START_STATE], []
    for i in range(total_frames):
        action                 = random.randint(0, N_ACTIONS - 1)
        next_state, _, done    = step_env(state, action)
        traj.append(next_state)
        if done:
            crashes.append(i + 1)
            state = START_STATE      # reset to start after crash
        else:
            state = next_state
    return traj, crashes

random_traj, crash_frames = run_random_agent(total_frames=80)
print(f"Random agent crashed {len(crash_frames)} times in {len(random_traj)} frames")
print(f"Crash frames: {crash_frames}")

# ── Animation ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8))
draw_track(ax, base_grid, "Before Training — Random Agent (crashes shown in orange)")
ax.legend(
    handles=[
        Patch(facecolor='#999999', label='Track'),
        Patch(facecolor='#44bb44', label='Start / Finish'),
        Patch(facecolor='#dd3333', label='Car'),
        Patch(facecolor='orange',  label='Crash / Reset'),
    ],
    loc='upper right', fontsize=10
)

trail_grid_r  = np.zeros((ROWS, COLS))
trail_im_r    = ax.imshow(np.zeros((ROWS, COLS)), cmap='Reds', vmin=0, vmax=1,
                           alpha=0.35, interpolation='nearest', aspect='equal')
car_marker_r  = ax.plot([], [], 's', color='#dd3333', markersize=16, zorder=5)[0]
quiver_r      = [None]
info_r        = ax.text(0.02, 0.97, '', transform=ax.transAxes, fontsize=10,
                         verticalalignment='top',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

def update_random(frame):
    r, c, heading, speed = random_traj[frame]
    is_crash = frame in crash_frames

    car_marker_r.set_data([c], [r])
    car_marker_r.set_color('orange' if is_crash else '#dd3333')

    if quiver_r[0] is not None:
        quiver_r[0].remove()
    dx, dy = DIR_ARROW[heading]
    quiver_r[0] = ax.quiver(c, r, dx, dy, color='white', scale=1, scale_units='xy',
                              angles='xy', width=0.007, headwidth=4, headlength=5, zorder=6)

    trail_grid_r[r, c] = min(1.0, trail_grid_r[r, c] + 0.25)
    trail_im_r.set_data(trail_grid_r)

    info_r.set_text(
        f'Step:    {frame+1} / {len(random_traj)}\n'
        f'Pos:     ({r}, {c})\n'
        f'Heading: {DIR_NAME[heading]}\n'
        f'Speed:   {speed}\n'
        f'{"💥 CRASH" if is_crash else ""}'
    )

anim_r = FuncAnimation(fig, update_random, frames=len(random_traj), interval=220, blit=False)
plt.close()
HTML(anim_r.to_jshtml())

### Training the Agent

Now we run Q-learning for 6,000 episodes. The agent starts by exploring randomly (ε = 1.0) and gradually shifts toward exploiting what it has learned. Watch the reward curve climb as it figures out how to stay on the track and complete laps.

In [ ]:
random.seed(0)
Q         = defaultdict(lambda: np.zeros(N_ACTIONS))
alpha, gamma, epsilon = 0.15, 0.95, 1.0
ep_rewards = []

for ep in range(6000):
    state     = START_STATE
    total_r   = 0.0
    lap_angle = 0.0

    for _ in range(300):
        action = random.randint(0, N_ACTIONS-1) if random.random() < epsilon \
                 else int(np.argmax(Q[state]))

        next_state, reward, done = step_env(state, action)
        total_r += reward

        old_a = np.arctan2(state[0]-CR,      state[1]-CC)
        new_a = np.arctan2(next_state[0]-CR, next_state[1]-CC)
        d = new_a - old_a
        if d >  np.pi: d -= 2*np.pi
        if d < -np.pi: d += 2*np.pi
        lap_angle += d

        if lap_angle >= 2 * np.pi:
            reward += 100
            done    = True

        Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
        state = next_state
        if done:
            break

    ep_rewards.append(total_r)
    epsilon = max(0.05, epsilon * 0.999)

window   = 150
smoothed = np.convolve(ep_rewards, np.ones(window)/window, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(smoothed, color='steelblue', linewidth=2)
plt.title(f'Q-Learning — Avg Reward per Episode (window={window})', fontsize=12)
plt.xlabel('Episode'); plt.ylabel('Total Reward')
plt.axhline(0, color='gray', linewidth=0.8, linestyle='--')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Final ε: {epsilon:.3f}  |  Last 100 ep avg reward: {np.mean(ep_rewards[-100:]):.1f}")

### After Training: Staying on the Track

Same car, same track, same 9 actions — but now the agent uses what it learned. It should complete a full lap without crashing, steering through corners instead of blundering off the edge.

In [ ]:
def run_greedy(max_steps=250):
    state, traj, lap_angle = START_STATE, [START_STATE], 0.0
    for _ in range(max_steps):
        action                  = int(np.argmax(Q[state]))
        next_state, _, done     = step_env(state, action)
        old_a = np.arctan2(state[0]-CR,      state[1]-CC)
        new_a = np.arctan2(next_state[0]-CR, next_state[1]-CC)
        d = new_a - old_a
        if d >  np.pi: d -= 2*np.pi
        if d < -np.pi: d += 2*np.pi
        lap_angle += d
        traj.append(next_state)
        state = next_state
        if done or lap_angle >= 2*np.pi:
            break
    return traj

trajectory = run_greedy()
print(f"Greedy lap completed in {len(trajectory)} steps")

fig, ax = plt.subplots(figsize=(13, 8))
draw_track(ax, base_grid, "After Training — Trained Agent Completing a Lap")
ax.legend(
    handles=[
        Patch(facecolor='#999999', label='Track'),
        Patch(facecolor='#44bb44', label='Start / Finish'),
        Patch(facecolor='#dd3333', label='Car'),
    ],
    loc='upper right', fontsize=10
)

trail_grid_t = np.zeros((ROWS, COLS))
trail_im_t   = ax.imshow(np.zeros((ROWS, COLS)), cmap='Greens', vmin=0, vmax=1,
                          alpha=0.4, interpolation='nearest', aspect='equal')
car_marker_t = ax.plot([], [], 's', color='#dd3333', markersize=16, zorder=5)[0]
quiver_t     = [None]
info_t       = ax.text(0.02, 0.97, '', transform=ax.transAxes, fontsize=10,
                        verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

def update_trained(frame):
    r, c, heading, speed = trajectory[frame]
    car_marker_t.set_data([c], [r])

    if quiver_t[0] is not None:
        quiver_t[0].remove()
    dx, dy = DIR_ARROW[heading]
    quiver_t[0] = ax.quiver(c, r, dx, dy, color='white', scale=1, scale_units='xy',
                              angles='xy', width=0.007, headwidth=4, headlength=5, zorder=6)

    trail_grid_t[r, c] = min(1.0, trail_grid_t[r, c] + 0.3)
    trail_im_t.set_data(trail_grid_t)

    info_t.set_text(
        f'Step:    {frame+1} / {len(trajectory)}\n'
        f'Pos:     ({r}, {c})\n'
        f'Heading: {DIR_NAME[heading]}\n'
        f'Speed:   {speed}'
    )

anim_t = FuncAnimation(fig, update_trained, frames=len(trajectory), interval=220, blit=False)
plt.close()
HTML(anim_t.to_jshtml())

## Summary

| Concept | What it is |
|---|---|
| **Activation function** | Non-linear transform applied to each neuron's output — enables deep networks to learn complex patterns |
| **Loss function** | Measures how wrong the model is — MSE for regression, cross-entropy for classification |
| **Gradient descent** | Iteratively nudge weights in the direction that reduces loss |
| **Accuracy / Precision / Recall / F1** | Human-readable metrics for evaluating a trained classifier |
| **Supervised learning** | Learn from labelled examples |
| **Unsupervised learning** | Find hidden structure in unlabelled data |
| **Reinforcement learning** | Learn by trial and error, guided by a reward signal |

---

**Next notebook:** Use cases — Computer Vision, NLP/NLU, Speech-to-Text, and Text-to-Speech. What does AI actually look like in the real world?